# Data Preparation for Minor League Analysis

## Purpose

Transform data from the 2024 AAA season to prepare for analysis of the challenge system. Raw data was obtained from the following GitHub repo:

https://github.com/armstjc/milb-data-repository/tree/main

The following transformations are completed:

1) Filter to data from June 25th, 2024 to the end of the 2024 season, as this is when the challenge system was universally adopted across AAA (https://www.mlb.com/news/triple-a-abs-challenge-system?msockid=13a1423ec59b64c529f456edc46065f5)

2)  Create id columns for each game, plate appearance and pitch to allow for easier analysis

3) Understand the pitch type column, and use this understanding to label whether a batter swung at a pitch (useful for challenge analysis)

4) Identify whether a pitch was challenged, and if so whether the challenge was successful.

## Output

`data/export_data/clean_milb_data.csv`

Cleaned-up data from the 2024 AAA season, with all original columns plus the additions as specified below:

1) `game_id`

2) `plate_appearance_id`

3) `pitch_id`

4) `swing`: 'swing' if the batter swung at the pitch; 'take' if the batter did not swing; 'unclear' if unclear from the data

5) `challenge`: 1 if the pitch was challenged; 0 if not. Note that we can only identify challenges on the last pitch of an at-bat.

6) `challenge_successful`: 'successful' if the challenge was successful; 'unsuccessful' if unsuccessful; 'no challenge' if the pitch was not challenged

## Next Steps

1) Clear up the definitions of unclear `type` values to reduce uncretainty in `swing` values

1) Determine whether a pitch was in the strike zone using objective measurements

# Import Libraries

In [84]:
import polars as pl
import glob
import os

# Load Data

In [85]:
jun = pl.read_csv('../data/raw_data/2024_6_aaa_pbp.csv')
jul = pl.read_csv('../data/raw_data/2024_7_aaa_pbp.csv').with_columns(pl.col('zone').cast(pl.Int64))
aug = pl.read_csv('../data/raw_data/2024_8_aaa_pbp.csv', ignore_errors = True)
# Fix data types
columns_to_cast = ['spin_dir', 'release_spin_rate', 'spin_axis']
aug = aug.with_columns([
    pl.col(col).cast(pl.Float64) for col in columns_to_cast
])
sep = pl.read_csv('../data/raw_data/2024_9_aaa_pbp.csv').with_columns(pl.col('zone').cast(pl.Int64))
pbp = pl.concat([jun, jul, aug, sep]).unique()
pbp = pbp.with_columns(
    pl.col("play_start_datetime").str.to_datetime('%Y-%m-%d %H:%M:%S%.3f')
)
# Filter to games starting June 25
pbp = pbp.filter(pl.col('play_start_datetime') >= pl.datetime(2024, 6, 25))
# Create a unique id for each pitch
pbp = pbp.with_row_index().rename({'index': 'pitch_id'})
# Add game_id column
game_id_lookup = (
    pbp
        .select(['game_date', 'home_team_org_id', 'away_team_org_id'])
        .unique()
        .with_row_index()
        .rename({'index': 'game_id'})
)
pbp = pbp.join(game_id_lookup, on = ['game_date', 'home_team_org_id', 'away_team_org_id'])
# Add plate_appearance_id column
plate_appearance_id_lookup = (
    pbp
        .select(['game_id', 'inning', 'inning_top_bot', 'batter', 'pitcher', 'outs_when_up'])
        .unique()
        .with_row_index()
        .rename({'index': 'plate_appearance_id'})
)
pbp = pbp.join(plate_appearance_id_lookup, on = ['game_id', 'inning', 'inning_top_bot', 'batter', 'pitcher', 'outs_when_up'])
# Add lag values for balls and strikes (count before the pitch is thrown)
pbp = (
    pbp
        .with_columns([
            pl.col('balls')
            .shift(1)
            .over(
                partition_by = ['game_id', 'plate_appearance_id'],
                order_by = 'pitch_number'
            )
            .fill_null(strategy = 'zero')
            .alias('lag_balls'),
            pl.col('strikes')
            .shift(1)
            .over(
                partition_by = ['game_id', 'plate_appearance_id'],
                order_by = 'pitch_number'
            )
            .fill_null(strategy = 'zero')
            .alias('lag_strikes')
        ]
        )
)
pbp.head()

pitch_id,play_start_datetime,play_end_datetime,pitch_type,pitch_name,game_date,release_speed,release_pos_x,release_pos_y,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,…,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,game_month,game_day,game_year,league_id,league_name,league_level_id,league_level_name,away_team_org_id,away_team_org_name,home_team_org_id,home_team_org_name,game_id,plate_appearance_id,lag_balls,lag_strikes
u32,datetime[ms],str,str,str,str,f64,f64,f64,f64,str,i64,i64,str,str,f64,str,str,str,i64,str,str,str,str,str,str,str,f64,str,i64,i64,f64,f64,f64,f64,f64,f64,…,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,f64,str,str,i64,i64,i64,i64,str,i64,str,i64,str,i64,str,u32,u32,i64,i64
0,2024-06-28 01:53:58.280,"""2024-06-28 01:54:02.789""","""CH""","""Changeup""","""2024-06-27""",86.6,-1.504636,50.000499,5.854567,"""Peter Lambert""",519303,663567,null,"""Elliot Soto strikes out on a f…",229.0,null,null,null,6,"""Elliot Soto strikes out on a f…","""R""","""R""","""R""","""ABQ""","""SL""","""C""",null,null,1,1,-6.941855,2.973652,0.753154,2.103768,null,null,…,null,null,null,null,null,null,null,34,2,6,1,1,6,1,6,1,6,null,null,229.0,null,null,6,27,2024,112,"""Pacific Coast League""",11,"""Triple-A""",108,"""Los Angeles Angels""",115,"""Colorado Rockies""",1036,22017,1,0
1,2024-09-18 00:19:15.010,"""2024-09-18 00:19:25.647""","""FF""","""Four-Seam Fastball""","""2024-09-17""",88.1,3.232269,50.004946,5.669619,"""Mason Fluharty""",666211,689254,null,"""Taylor Trammell called out on …",169.0,null,null,null,4,"""Taylor Trammell called out on …","""R""","""L""","""L""","""BUF""","""SWB""","""B""",null,null,1,0,-2.407521,4.213267,-0.760974,2.705338,null,null,…,null,null,null,null,null,null,null,58,1,3,3,3,3,3,3,3,3,null,null,169.0,null,null,9,17,2024,117,"""International League""",11,"""Triple-A""",147,"""New York Yankees""",141,"""Toronto Blue Jays""",807,22994,0,0
2,2024-07-25 03:06:15.107,"""2024-07-25 03:06:19.295""","""SL""","""Slider""","""2024-07-24""",85.0,0.673434,50.003952,5.825734,"""Blake Taylor""",680862,642130,null,"""Willie MacIver strikes out on …",288.0,null,null,null,13,"""Willie MacIver strikes out on …","""R""","""R""","""L""","""ABQ""","""RR""","""B""",null,null,2,2,-1.537159,-0.80345,-1.014508,0.475927,null,null,…,null,null,null,null,null,null,null,73,7,4,7,4,7,7,4,4,7,null,null,288.0,null,null,7,24,2024,112,"""Pacific Coast League""",11,"""Triple-A""",140,"""Texas Rangers""",115,"""Colorado Rockies""",943,66672,1,2
3,2024-08-03 22:40:47.492,"""2024-08-03 22:40:51.501""","""FF""","""Four-Seam Fastball""","""2024-08-03""",95.2,-0.00569,50.002266,6.428304,"""Braydon Fisher""",681508,680755,null,"""Mickey Gasper walks. Triston…",203.0,null,null,null,11,"""Mickey Gasper walks. Triston…","""R""","""L""","""R""","""WOR""","""BUF""","""B""",null,null,2,2,-3.876377,10.492538,-0.363394,4.554971,805367.0,671213.0,…,null,null,null,null,null,null,null,71,4,9,5,9,5,5,9,9,5,null,null,203.0,null,null,8,3,2024,117,"""International League""",11,"""Triple-A""",141,"""Toronto Blue Jays""",111,"""Boston Red Sox""",615,24923,1,2
4,2024-08-16 23:05:44.932,"""2024-08-16 23:05:51.595""","""CU""","""Curveball""","""2024-08-16""",74.2,-1.255317,50.000262,5.939749,"""Carlos Rodriguez""",682927,692230,null,"""Ronny Simon grounds out, first…",47.0,null,null,null,13,"""Ronny Simon grounds out, first…","""R""","""L""","""R""","""DUR""","""NAS""","""B""",null,null,2,2,3.979927,-5.003311,-1.287785,1.722

# Add Swing Indicator

## Understanding the Type Column

In [86]:
# 17 different type values
with pl.Config(tbl_rows = 17):
    print(pbp.group_by(pl.col('type')).len().sort('len', descending = True))

shape: (17, 2)
┌──────┬────────┐
│ type ┆ len    │
│ ---  ┆ ---    │
│ str  ┆ u32    │
╞══════╪════════╡
│ B    ┆ 118950 │
│ F    ┆ 59104  │
│ C    ┆ 51821  │
│ S    ┆ 37706  │
│ X    ┆ 35162  │
│ D    ┆ 13231  │
│ E    ┆ 8319   │
│ *B   ┆ 6597   │
│ T    ┆ 3351   │
│ W    ┆ 1622   │
│ H    ┆ 1064   │
│ L    ┆ 681    │
│ M    ┆ 116    │
│ O    ┆ 19     │
│ P    ┆ 18     │
│ R    ┆ 1      │
│ Z    ┆ 1      │
└──────┴────────┘


### B

ball -- labeled in data dictionary

### F

foul ball

In [87]:
# always a strike 
(
    pbp
        .filter(pl.col('type') == 'F')
        .with_columns(
            pl.when(pl.col('strikes') >= pl.col('lag_strikes'))
            .then(1)
            .otherwise(0)
            .alias('strike')
        )
        .group_by('strike')
        .len()
)

strike,len
i32,u32
1,59104


In [88]:
# when there's two strikes, the count doesn't change
def view_ball_strike_count(df, type):
    """
    Print the balls and strikes, as well as lag values, for each pitch to inspect the implication of the type
    """
    result = (
        df
            .filter(pl.col('type') == type)
            .select([
                'balls',
                'strikes',
                'lag_balls',
                'lag_strikes'
                ])
            .head(10)
    )
    print(result)
view_ball_strike_count(pbp.filter(pl.col('lag_strikes') == 2), 'F')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 2     ┆ 2       ┆ 2         ┆ 2           │
│ 1     ┆ 2       ┆ 1         ┆ 2           │
│ 1     ┆ 2       ┆ 1         ┆ 2           │
│ 0     ┆ 2       ┆ 0         ┆ 2           │
│ 1     ┆ 2       ┆ 1         ┆ 2           │
│ 2     ┆ 2       ┆ 2         ┆ 2           │
│ 2     ┆ 2       ┆ 2         ┆ 2           │
│ 0     ┆ 2       ┆ 0         ┆ 2           │
│ 1     ┆ 2       ┆ 1         ┆ 2           │
│ 3     ┆ 2       ┆ 3         ┆ 2           │
└───────┴─────────┴───────────┴─────────────┘


### C

Called strike

In [89]:
# always a strike 
(
    pbp
        .filter(pl.col('type') == 'C')
        .with_columns(
            pl.when(pl.col('strikes') > pl.col('lag_strikes'))
            .then(1)
            .otherwise(0)
            .alias('strike')
        )
        .group_by('strike')
        .len()
)

strike,len
i32,u32
1,51821


In [90]:
# when strike three, the description shows it's a called strike
def view_description(df, type):
    """
    Outputs the count of the last pitch of the plate appearance (assuming walk or K) and accompanying description of the play
    """
    result = (
        df
            .filter(
                (pl.col('type') == type) &
                (
                (pl.col('strikes') == 3) |
                (pl.col('balls') == 4)
                )
            )
            .select('des')
            .to_series()
            .to_list()
            [:10]
    )
    return result
view_description(pbp, 'C')

['Sandy León called out on strikes.',
 'Kyle Manzardo called out on strikes.',
 'Diego Cartaya called out on strikes.',
 'Andrew Pinckney called out on strikes.',
 'Bobby Dalbec called out on strikes.',
 'Dom Nuñez called out on strikes.',
 'Neyfy Castillo called out on strikes.',
 'Michael Stefanic called out on strikes.',
 'Jonatan Clase called out on strikes.',
 'Danny Mendick called out on strikes.']

### S

Swinging strike

In [91]:
# Always a strike
view_ball_strike_count(pbp, 'S')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 3     ┆ 3       ┆ 3         ┆ 2           │
│ 2     ┆ 3       ┆ 2         ┆ 2           │
│ 1     ┆ 1       ┆ 1         ┆ 0           │
│ 1     ┆ 3       ┆ 1         ┆ 2           │
│ 1     ┆ 3       ┆ 1         ┆ 2           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 2     ┆ 3       ┆ 2         ┆ 2           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 2     ┆ 1       ┆ 2         ┆ 0           │
└───────┴─────────┴───────────┴─────────────┘


In [92]:
# Description indicate swinging strikes
view_description(pbp, 'S')

['José Fermín strikes out swinging.',
 'Nate Mondou strikes out swinging.',
 'Justice Bigbie strikes out swinging.',
 'Bryce Teodosio strikes out swinging.',
 'Cristopher Navarro strikes out swinging.',
 'Rodolfo Castro strikes out swinging.',
 'Jasson Domínguez strikes out swinging.',
 'Andrew Pinckney strikes out swinging.',
 'Jackson Holliday strikes out swinging.',
 'Agustin Ramirez strikes out swinging.']

### X

Ball in play (data dictionary)

### D, E

Balls in play

In [93]:
# count stays the same
view_ball_strike_count(pbp, 'D')
view_ball_strike_count(pbp, 'E')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 1     ┆ 1       ┆ 1         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 1           │
│ 0     ┆ 2       ┆ 0         ┆ 2           │
│ 0     ┆ 2       ┆ 0         ┆ 2           │
│ 0     ┆ 0       ┆ 0         ┆ 0           │
│ 0     ┆ 2       ┆ 0         ┆ 2           │
│ 1     ┆ 1       ┆ 1         ┆ 1           │
│ 0     ┆ 2       ┆ 0         ┆ 2           │
│ 0     ┆ 1       ┆ 0         ┆ 1           │
│ 0     ┆ 0       ┆ 0         ┆ 0           │
└───────┴─────────┴───────────┴─────────────┘
shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 2 

### *B

Ball

In [94]:
view_ball_strike_count(pbp, '*B')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 3     ┆ 2       ┆ 2         ┆ 2           │
│ 1     ┆ 1       ┆ 0         ┆ 1           │
│ 2     ┆ 0       ┆ 1         ┆ 0           │
│ 3     ┆ 1       ┆ 2         ┆ 1           │
│ 2     ┆ 2       ┆ 1         ┆ 2           │
│ 1     ┆ 0       ┆ 0         ┆ 0           │
│ 4     ┆ 2       ┆ 3         ┆ 2           │
│ 2     ┆ 1       ┆ 1         ┆ 1           │
│ 3     ┆ 2       ┆ 2         ┆ 2           │
│ 2     ┆ 1       ┆ 1         ┆ 1           │
└───────┴─────────┴───────────┴─────────────┘


### T

Foul tip

In [95]:
# always a strike
view_ball_strike_count(pbp, 'T')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 3     ┆ 3       ┆ 3         ┆ 2           │
│ 1     ┆ 3       ┆ 1         ┆ 2           │
│ 2     ┆ 1       ┆ 2         ┆ 0           │
│ 1     ┆ 2       ┆ 1         ┆ 1           │
│ 2     ┆ 2       ┆ 2         ┆ 1           │
│ 0     ┆ 3       ┆ 0         ┆ 2           │
│ 1     ┆ 1       ┆ 1         ┆ 0           │
│ 1     ┆ 2       ┆ 1         ┆ 1           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 2     ┆ 3       ┆ 2         ┆ 2           │
└───────┴─────────┴───────────┴─────────────┘


In [96]:
view_description(pbp, 'T')

['Donovan Walton strikes out on a foul tip.',
 'Gabriel Cancel strikes out on a foul tip.',
 'Kevin Smith strikes out on a foul tip.',
 'Drew Avans strikes out on a foul tip.',
 'Owen Caissie strikes out on a foul tip.',
 'Jeter Downs strikes out on a foul tip.',
 'Daniel Johnson strikes out on a foul tip.',
 'César Prieto strikes out on a foul tip.',
 'Kobe Kato strikes out on a foul tip.',
 'Jake Hager strikes out on a foul tip.']

### W

Swinging strike -- maybe 'whiff'?

In [97]:
view_ball_strike_count(pbp, 'W')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 1     ┆ 2       ┆ 1         ┆ 1           │
│ 0     ┆ 3       ┆ 0         ┆ 2           │
│ 0     ┆ 3       ┆ 0         ┆ 2           │
│ 0     ┆ 3       ┆ 0         ┆ 2           │
│ 1     ┆ 3       ┆ 1         ┆ 2           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 3     ┆ 2       ┆ 3         ┆ 1           │
│ 0     ┆ 3       ┆ 0         ┆ 2           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
└───────┴─────────┴───────────┴─────────────┘


In [98]:
view_description(pbp, 'W')

["Brian O'Keefe strikes out swinging, catcher Nick Raposo to first baseman Joey Votto.",
 'Jonatan Clase strikes out swinging.',
 'Bryce Teodosio strikes out swinging.',
 'José Azocar strikes out swinging.',
 'Jonatan Clase strikes out swinging.',
 'Jose Sanabria strikes out swinging.',
 'Thomas Saggese strikes out swinging.',
 'Josh Kasevich strikes out swinging, catcher Drake Baldwin to first baseman Brian Anderson.',
 'Abraham Toro strikes out swinging.',
 'Kyren Paris strikes out swinging.']

### H

Hit by pitch

In [99]:
view_description(pbp, 'H')

['Nick Maton hit by pitch.',
 'Seth Beer hit by pitch.    Joshua Palacios to 3rd.    Matt Gorski to 2nd.',
 'Diego A.   Castillo hit by pitch.',
 'Sandy León hit by pitch.',
 'Hudson Haskin hit by pitch.',
 'Thomas Saggese hit by pitch.',
 'Wilmer Difo hit by pitch.',
 'Andrew Navigato hit by pitch.',
 'Dustin Harris hit by pitch.    Blaine Crim to 2nd.',
 'Bobby Dalbec hit by pitch.']

### L

foul bunt

In [100]:
# always a strike
view_ball_strike_count(pbp, 'L')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 1     ┆ 2       ┆ 1         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 1     ┆ 2       ┆ 1         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
└───────┴─────────┴───────────┴─────────────┘


In [101]:
view_description(pbp, 'L')

['Bryce Johnson strikes out on a foul bunt.',
 'Anthony Servideo strikes out on a foul bunt.',
 'Austin Nola strikes out on a foul bunt.',
 'Livan Soto strikes out on a foul bunt.']

### M
Definitely a strike, but unclear whether the batter swung or not

In [102]:
# definitely a strike
view_ball_strike_count(pbp, 'M')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
└───────┴─────────┴───────────┴─────────────┘


In [103]:
# never a strikeout
view_description(pbp, 'M')

[]

### O

Similar to M -- definitely a strike, but unknown if the batter swung or not

In [104]:
# definitely a strike
view_ball_strike_count(pbp, 'O')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
│ 2     ┆ 1       ┆ 2         ┆ 0           │
│ 0     ┆ 2       ┆ 0         ┆ 1           │
│ 0     ┆ 1       ┆ 0         ┆ 0           │
└───────┴─────────┴───────────┴─────────────┘


In [105]:
# never a strikeout
view_description(pbp, 'O')

[]

### P

Ball, but never a walk

In [106]:
view_ball_strike_count(pbp, 'P')

shape: (10, 4)
┌───────┬─────────┬───────────┬─────────────┐
│ balls ┆ strikes ┆ lag_balls ┆ lag_strikes │
│ ---   ┆ ---     ┆ ---       ┆ ---         │
│ i64   ┆ i64     ┆ i64       ┆ i64         │
╞═══════╪═════════╪═══════════╪═════════════╡
│ 2     ┆ 0       ┆ 1         ┆ 0           │
│ 1     ┆ 0       ┆ 0         ┆ 0           │
│ 1     ┆ 0       ┆ 0         ┆ 0           │
│ 1     ┆ 0       ┆ 0         ┆ 0           │
│ 1     ┆ 1       ┆ 0         ┆ 1           │
│ 2     ┆ 2       ┆ 1         ┆ 2           │
│ 1     ┆ 0       ┆ 0         ┆ 0           │
│ 1     ┆ 0       ┆ 0         ┆ 0           │
│ 1     ┆ 0       ┆ 0         ┆ 0           │
│ 1     ┆ 1       ┆ 0         ┆ 1           │
└───────┴─────────┴───────────┴─────────────┘


In [107]:
view_description(pbp, 'P')

[]

### Z and R
one-off data errors -- unclear

In [108]:
# always a strike
(
    pbp
        .filter(pl.col('type').is_in(['Z', 'R']))
        .select([
            'balls',
            'strikes',
            'lag_balls',
            'lag_strikes',
            'type',
            'des'
            ])
)

balls,strikes,lag_balls,lag_strikes,type,des
i64,i64,i64,i64,str,str
2,2,2,2,"""Z""","""Jordan Diaz singles on a groun…"
1,2,1,1,"""R""","""Phil Clarke walks. Damiano P…"


## Create Swing Indicator Column

In [109]:
# counting bunt as a swing
swing = ['F', 'S', 'X', 'D', 'E', 'T', 'W', 'L']
take = ['B', 'C', '*B', 'H', 'P']
unclear = ['M', 'O', 'R', 'Z']
pbp = pbp.with_columns(
    pl.when(pl.col('type').is_in(swing))
    .then(pl.lit('swing'))
    .when(pl.col('type').is_in(take))
    .then(pl.lit('take'))
    .when(pl.col('type').is_in(unclear))
    .then(pl.lit('unclear'))
    .otherwise(pl.lit('error'))
    .alias('swing')
)
pbp.select(pl.col('swing').value_counts())

swing
struct[2]
"{""swing"",159176}"
"{""take"",178450}"
"{""unclear"",137}"


# Add Challenge Indicator

IMPORTANT NOTE: Due to the structure of the data, we can only identify whether a challenge occurred on the last pitch of the plate appearance (resulting in a strikeout or walk). Therefore, some pitches that were challenged will not be labeled as such in our data

In [110]:
# Create a lookup table of challenges
last_pitch = (
    pbp
        .group_by('plate_appearance_id', 'des')
        .agg(pl.col('pitch_number').max().alias('last_pitch_number'))
        .with_columns([
            pl.when(pl.col('des').str.contains('challenge'))
            .then(1)
            .otherwise(0)
            .alias('challenge'),

            pl.when(
                (pl.col('des').str.contains('challenge')) &
                (pl.col('des').str.contains(r"(?i)overturned"))
            )
            .then(pl.lit('successful'))
            .when(
                (pl.col('des').str.contains('challenge')) &
                (pl.col('des').str.contains(r"(?i)upheld"))
            )
            .then(pl.lit('unsuccessful'))
            .when(~(pl.col('des').str.contains('challenge')))
            .then(pl.lit('no challenge'))
            .otherwise(pl.lit('error'))
            .alias('challenge_successful')
            ])
)
# Join back into main dataset
pbp = (
    pbp
        .join(
            last_pitch,
            left_on = ['plate_appearance_id', 'pitch_number', 'des'],
            right_on = ['plate_appearance_id', 'last_pitch_number', 'des'],
            how = 'left')
        .with_columns([
            pl.col('challenge').fill_null(strategy = 'zero'),

            pl.col('challenge_successful').fill_null('no challenge')
        ])
)

# Export to CSV

In [114]:
pbp.write_csv('../data/export_data/aaa_pbp.csv')